# Linear Regression Trajectory Regularizer Estimation

This notebook implements two variants of the early-stopping experiment from Figure 3 and Appendix B:
1. Learn a diagonal regularizer from a **single trajectory** using multiple steps (with burn-in).
2. Learn a diagonal regularizer from **10 independent runs**, using only endpoint gradients.

It also plots distance to the theoretical regularizer as a function of trajectory length and number of runs.

In [1]:
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)

ROOT = Path.cwd()
if not (ROOT / "figures").exists():
    ROOT = ROOT.parent
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

style_path = ROOT / "clean_fig.mplstyle"
if style_path.exists():
    plt.style.use(style_path)

SEED = 56
torch.manual_seed(SEED)
np.random.seed(SEED)

ROOT, FIG_DIR

(PosixPath('/home/kevin/OneDrive/Documents/UMich/Research/inductive-bias'),
 PosixPath('/home/kevin/OneDrive/Documents/UMich/Research/inductive-bias/figures'))

In [2]:
def mse_grad(X: torch.Tensor, y: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    n = X.shape[0]
    return X.T @ (X @ theta - y) / n


def gd_trajectory(X: torch.Tensor, y: torch.Tensor, steps: int, lr: float):
    p = X.shape[1]
    theta = torch.zeros(p, dtype=X.dtype, device=X.device)
    thetas = [theta.clone()]
    for _ in range(steps):
        theta = theta - lr * mse_grad(X, y, theta)
        thetas.append(theta.clone())
    thetas = torch.stack(thetas)
    grads = torch.stack([mse_grad(X, y, t) for t in thetas])
    b = -grads
    return thetas, b


def compute_q_matrix(X: torch.Tensor, steps: int, lr: float) -> torch.Tensor:
    if steps <= 0:
        raise ValueError("steps must be >= 1")
    n = X.shape[0]
    A = X.T @ X / n
    evals, evecs = torch.linalg.eigh(A)
    if torch.any(1 - lr * evals <= 0):
        raise ValueError("learning rate too large for stable inverse dynamics")
    d = evals / (torch.pow(1 - lr * evals, -steps) - 1.0)
    return evecs @ torch.diag(d) @ evecs.T


def ridge_closed_form_diag(X: torch.Tensor, y: torch.Tensor, lam_diag: torch.Tensor) -> torch.Tensor:
    n = X.shape[0]
    lhs = X.T @ X + n * torch.diag(lam_diag)
    rhs = X.T @ y
    return torch.linalg.solve(lhs, rhs)


def fit_diag_from_points(theta_points: torch.Tensor, b_points: torch.Tensor) -> torch.Tensor:
    numer = (theta_points * b_points).sum(dim=0)
    denom = (theta_points.square()).sum(dim=0).clamp_min(1e-12)
    return numer / denom


def fit_diag_from_trajectory(
    thetas: torch.Tensor,
    b: torch.Tensor,
    steps_used: int,
    burn_in: int = 0,
) -> torch.Tensor:
    if steps_used <= burn_in:
        raise ValueError("steps_used must be > burn_in")
    return fit_diag_from_points(
        thetas[burn_in + 1 : steps_used + 1],
        b[burn_in + 1 : steps_used + 1],
    )


def fit_diag_from_endpoints(theta_endpoints: torch.Tensor, b_endpoints: torch.Tensor, runs_used: int) -> torch.Tensor:
    return fit_diag_from_points(theta_endpoints[:runs_used], b_endpoints[:runs_used])


def relative_distance(est: torch.Tensor, theory: torch.Tensor) -> float:
    return (torch.linalg.norm(est - theory) / torch.linalg.norm(theory)).item()


def plot_figure3_variant(
    lam_est: torch.Tensor,
    Q_theory: torch.Tensor,
    X: torch.Tensor,
    y: torch.Tensor,
    theta_hat: torch.Tensor,
    theta_true: torch.Tensor,
    out_path: Path,
    title: str,
):
    Q_est = torch.diag(lam_est)
    theta_explicit = ridge_closed_form_diag(X, y, lam_est)
    weights = torch.stack([theta_explicit, theta_hat, theta_true], dim=1).cpu().numpy()

    Q_est_np = Q_est.cpu().numpy()
    Q_theory_np = Q_theory.cpu().numpy()
    vmax_q = max(np.abs(Q_est_np).max(), np.abs(Q_theory_np).max())
    wmax = np.abs(weights).max()

    fig = plt.figure(figsize=(14, 5.8), constrained_layout=True)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.1, 0.8])

    ax0 = fig.add_subplot(gs[0, 0])
    im0 = ax0.imshow(Q_est_np, cmap="bwr", vmin=-vmax_q, vmax=vmax_q, aspect="equal")
    ax0.set_title(r"Estimated $\hat{\Lambda}$")
    ax0.set_xlabel(r"$p$")
    ax0.set_ylabel(r"$p$")

    ax1 = fig.add_subplot(gs[0, 1])
    ax1.imshow(Q_theory_np, cmap="bwr", vmin=-vmax_q, vmax=vmax_q, aspect="equal")
    ax1.set_title(r"Theoretical $\Lambda$")
    ax1.set_xlabel(r"$p$")

    cbar0 = fig.colorbar(im0, ax=[ax0, ax1], shrink=0.9, pad=0.02)
    cbar0.ax.set_ylabel("value")

    ax2 = fig.add_subplot(gs[0, 2])
    im2 = ax2.imshow(weights, cmap="bwr", vmin=-wmax, vmax=wmax, aspect="equal")
    ax2.set_title(r"$\hat{	heta}_{\Lambda}$ vs $\hat{	heta}$ vs $	heta$")
    ax2.set_xticks([0, 1, 2])
    ax2.set_xticklabels([r"$\hat{	heta}_{\Lambda}$", r"$\hat{	heta}$", r"$	heta$"])
    ax2.set_ylabel(r"$p$")
    fig.colorbar(im2, ax=ax2, shrink=0.9, pad=0.02)

    fig.suptitle(title)
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

    return theta_explicit


In [3]:
N = 1000
P = 10
LR = 1e-2
T = 500
NUM_RUNS = 10
BURN_IN = 50  # steps discarded in the single-trajectory fit

X = torch.randn(N, P)
Q_theory = compute_q_matrix(X, T, LR)
lambda_theory = torch.diag(Q_theory)

run_data = []
for run_idx in range(NUM_RUNS):
    g = torch.Generator().manual_seed(SEED + 100 + run_idx)
    beta = 3.0 * torch.randn(P, generator=g)
    y = X @ beta
    thetas, b = gd_trajectory(X, y, T, LR)
    run_data.append({"beta": beta, "y": y, "thetas": thetas, "b": b})

single = run_data[0]
theta_single = single["thetas"]
b_single = single["b"]
y_single = single["y"]
beta_single = single["beta"]

theta_endpoints = torch.stack([r["thetas"][-1] for r in run_data])
b_endpoints = torch.stack([r["b"][-1] for r in run_data])

print(f"Theory diag norm: {torch.linalg.norm(lambda_theory).item():.6g}")
print(f"Endpoint tensor shape: {theta_endpoints.shape}")
print(f"Single-trajectory burn-in: {BURN_IN} steps")


Theory diag norm: 0.0267822
Endpoint tensor shape: torch.Size([10, 10])
Single-trajectory burn-in: 50 steps


## 1) Appendix B single-trajectory fit

Fit one shared diagonal regularizer using one trajectory after an initial burn-in period, then instantiate the Figure 3-style comparison at `m = T`.

In [4]:
lambda_traj = fit_diag_from_trajectory(theta_single, b_single, T, burn_in=BURN_IN)
traj_dist_full = relative_distance(lambda_traj, lambda_theory)

fig_single_path = FIG_DIR / "linear_regression_trajectory_fig3_single_trajectory.pdf"
_ = plot_figure3_variant(
    lam_est=lambda_traj,
    Q_theory=Q_theory,
    X=X,
    y=y_single,
    theta_hat=theta_single[-1],
    theta_true=beta_single,
    out_path=fig_single_path,
    title=f"Figure 3 Variant: Single-Trajectory Fit (burn-in={BURN_IN})",
)

print(f"Single-trajectory relative distance (burn-in={BURN_IN}, m=T): {traj_dist_full:.6f}")
fig_single_path


Single-trajectory relative distance (burn-in=50, m=T): 15.775502


PosixPath('/home/kevin/OneDrive/Documents/UMich/Research/inductive-bias/figures/linear_regression_trajectory_fig3_single_trajectory.pdf')

## 2) Endpoint-only fit across 10 runs

Fit one shared diagonal regularizer from endpoint equations only, stacking 10 independent runs.

In [5]:
lambda_runs10 = fit_diag_from_endpoints(theta_endpoints, b_endpoints, NUM_RUNS)
runs_dist_full = relative_distance(lambda_runs10, lambda_theory)

fig_runs_path = FIG_DIR / "linear_regression_trajectory_fig3_10run_endpoint.pdf"
_ = plot_figure3_variant(
    lam_est=lambda_runs10,
    Q_theory=Q_theory,
    X=X,
    y=y_single,
    theta_hat=theta_single[-1],
    theta_true=beta_single,
    out_path=fig_runs_path,
    title="Figure 3 Variant: 10-Run Endpoint-Only Fit",
)

print(f"10-run endpoint relative distance: {runs_dist_full:.6f}")
fig_runs_path

10-run endpoint relative distance: 0.195796


PosixPath('/home/kevin/OneDrive/Documents/UMich/Research/inductive-bias/figures/linear_regression_trajectory_fig3_10run_endpoint.pdf')

## 3) Distance to theory curves

Left: distance vs total number of steps for method (1), using a fixed burn-in.

Right: distance vs number of runs for method (2).

Both y-axes are log-scaled to avoid visual compression.

In [6]:
step_counts = torch.arange(BURN_IN + 1, T + 1)
traj_distances = torch.empty(len(step_counts), dtype=torch.float64)
for i, m in enumerate(step_counts.tolist()):
    lam_m = fit_diag_from_trajectory(theta_single, b_single, m, burn_in=BURN_IN)
    traj_distances[i] = relative_distance(lam_m, lambda_theory)

run_counts = torch.arange(1, NUM_RUNS + 1)
run_distances = torch.empty(NUM_RUNS, dtype=torch.float64)
for i, k in enumerate(run_counts.tolist()):
    lam_k = fit_diag_from_endpoints(theta_endpoints, b_endpoints, k)
    run_distances[i] = relative_distance(lam_k, lambda_theory)

fig_dist_path = FIG_DIR / "linear_regression_trajectory_distance_to_theory.pdf"

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), constrained_layout=True)

axes[0].semilogy(step_counts.numpy(), traj_distances.numpy(), color="#1f77b4")
axes[0].set_title("Single Trajectory")
axes[0].set_xlabel("Total Steps m (fit on burn-in+1..m)")
axes[0].set_ylabel("Relative Distance to Theory")
axes[0].grid(alpha=0.3, which="both")

axes[1].semilogy(run_counts.numpy(), run_distances.numpy(), marker="o", color="#d62728")
axes[1].set_title("Endpoint-Only Across Runs")
axes[1].set_xlabel("Number of Runs Used")
axes[1].set_ylabel("Relative Distance to Theory")
axes[1].grid(alpha=0.3, which="both")

fig.suptitle("Distance from Theoretical Regularizer")
fig.savefig(fig_dist_path, bbox_inches="tight")
plt.close(fig)

print(f"Trajectory distance at m={step_counts[0].item()} (first post burn-in): {traj_distances[0].item():.6f}")
print(f"Trajectory distance at m=T: {traj_distances[-1].item():.6f}")
print(f"Run distance at k=1: {run_distances[0].item():.6f}")
print(f"Run distance at k={NUM_RUNS}: {run_distances[-1].item():.6f}")
fig_dist_path


Trajectory distance at m=51 (first post burn-in): 168.419872
Trajectory distance at m=T: 15.775502
Run distance at k=1: 18.674916
Run distance at k=10: 0.195796


PosixPath('/home/kevin/OneDrive/Documents/UMich/Research/inductive-bias/figures/linear_regression_trajectory_distance_to_theory.pdf')

In [7]:
print("Generated files:")
print(fig_single_path)
print(fig_runs_path)
print(fig_dist_path)

Generated files:
/home/kevin/OneDrive/Documents/UMich/Research/inductive-bias/figures/linear_regression_trajectory_fig3_single_trajectory.pdf
/home/kevin/OneDrive/Documents/UMich/Research/inductive-bias/figures/linear_regression_trajectory_fig3_10run_endpoint.pdf
/home/kevin/OneDrive/Documents/UMich/Research/inductive-bias/figures/linear_regression_trajectory_distance_to_theory.pdf
